# MISION: Certificar el Detector

**Objetivo:** Evaluacion rigurosa y diseno de un sistema de alertas operativo. La pregunta final: confiarias tu vida a este modelo?

---

### Agenda
1. NASA Scoring Function detallada
2. Comparacion final de todos los modelos
3. Analisis por rango de RUL: precision donde importa
4. Sistema de alertas: verde/amarillo/rojo
5. Simulacion operativa
6. Resumen ejecutivo y arquitectura de produccion

## 1. Imports y Carga

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras

plt.style.use('seaborn-v0_8-whitegrid')
HEALTHY = '#2ecc71'
DEGRADED = '#f39c12'
CRITICAL = '#e74c3c'
NEUTRAL = '#3498db'

import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
processed_dir = os.path.join(project_root, "data", "cmapss", "processed")
models_dir = os.path.join(project_root, "models")

In [ ]:
# Cargar datos
df_test = pd.read_parquet(os.path.join(processed_dir, 'test_features.parquet'))
df_test_last = pd.read_parquet(os.path.join(processed_dir, 'test_last_cycle.parquet'))

# Cargar modelos
xgb_model = joblib.load(os.path.join(models_dir, 'rul_xgboost.pkl'))
rf_model = joblib.load(os.path.join(models_dir, 'rul_random_forest.pkl'))
lstm_model = keras.models.load_model(os.path.join(models_dir, 'rul_lstm.keras'))

metadata = joblib.load(os.path.join(models_dir, 'rul_predictor_metadata.pkl'))
feature_cols = metadata['feature_cols']
WINDOW_SIZE = metadata.get('window_size', 30)

print(f"Test data: {df_test.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Window size: {WINDOW_SIZE}")

## 2. NASA Scoring Function — En Detalle

La NASA diseno una funcion de scoring asimetrica para C-MAPSS:

$$S = \sum_{i=1}^{n} s_i$$

Donde para cada motor $i$, con $d_i = RUL_{predicho} - RUL_{real}$:

- Si $d_i < 0$ (prediccion temprana/conservadora): $s_i = e^{-d_i/13} - 1$
- Si $d_i \geq 0$ (prediccion tardia/optimista): $s_i = e^{d_i/10} - 1$

**Interpretacion critica:**
- Predecir **temprano** (decir que al motor le quedan menos ciclos de los reales): costo moderado. Se reemplaza el motor antes de tiempo — desperdicio economico pero seguro.
- Predecir **tarde** (decir que le quedan MAS ciclos de los reales): costo exponencial. El motor puede fallar antes de la inspeccion programada — catastrofe.

In [ ]:
def nasa_score(y_true, y_pred):
    """NASA Scoring Function."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(scores)

def nasa_score_per_sample(y_true, y_pred):
    """NASA Score por muestra individual."""
    d = y_pred - y_true
    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)

# Visualizar la asimetria de la funcion
d_range = np.linspace(-50, 50, 1000)
s_values = np.where(d_range < 0, np.exp(-d_range / 13) - 1, np.exp(d_range / 10) - 1)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(d_range[d_range < 0], s_values[d_range < 0], color=NEUTRAL, linewidth=3, label='Early (d < 0): conservador')
ax.plot(d_range[d_range >= 0], s_values[d_range >= 0], color=CRITICAL, linewidth=3, label='Late (d >= 0): peligroso')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_title('NASA Scoring Function: Asimetria del Costo', fontsize=14, fontweight='bold')
ax.set_xlabel('d = RUL_predicho - RUL_real')
ax.set_ylabel('Score (penalizacion)')
ax.legend(fontsize=12)
ax.set_ylim(0, 100)
ax.set_xlim(-50, 50)

# Anotar
ax.annotate('Predecir tarde\n(PELIGROSO)', xy=(30, 50), fontsize=11, color=CRITICAL, fontweight='bold', ha='center')
ax.annotate('Predecir temprano\n(costoso pero seguro)', xy=(-30, 15), fontsize=11, color=NEUTRAL, fontweight='bold', ha='center')

plt.tight_layout()
plt.show()

## 3. Comparacion Final de Todos los Modelos

In [ ]:
# Predicciones de cada modelo
X_test_classic = df_test_last[feature_cols].values
y_test_vals = df_test_last['RUL'].values

y_pred_xgb = xgb_model.predict(X_test_classic)
y_pred_rf = rf_model.predict(X_test_classic)

# LSTM: secuencias del ultimo ciclo
def get_last_sequences(df, feature_cols, window_size):
    X_last, y_last = [], []
    for unit in sorted(df['unit'].unique()):
        unit_data = df[df['unit'] == unit].sort_values('time_cycles')
        features = unit_data[feature_cols].values
        rul = unit_data['RUL'].values
        if len(features) < window_size:
            pad_length = window_size - len(features)
            features = np.vstack([np.tile(features[0], (pad_length, 1)), features])
            rul = np.concatenate([np.full(pad_length, rul[0]), rul])
        X_last.append(features[-window_size:])
        y_last.append(rul[-1])
    return np.array(X_last), np.array(y_last)

X_test_lstm, y_test_lstm = get_last_sequences(df_test, feature_cols, WINDOW_SIZE)
y_pred_lstm = lstm_model.predict(X_test_lstm, verbose=0).flatten()

# Tabla comparativa
models = {
    'Random Forest': (y_test_vals, y_pred_rf),
    'XGBoost': (y_test_vals, y_pred_xgb),
    'LSTM': (y_test_lstm, y_pred_lstm)
}

print("COMPARACION FINAL")
print("=" * 75)
print(f"  {'Modelo':<25} {'RMSE':>8} {'MAE':>8} {'R2':>8} {'NASA Score':>12}")
print("-" * 75)

comparison_data = []
for name, (yt, yp) in models.items():
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae = mean_absolute_error(yt, yp)
    r2 = r2_score(yt, yp)
    ns = nasa_score(yt, yp)
    comparison_data.append({'Modelo': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'NASA_Score': ns})
    print(f"  {name:<25} {rmse:>8.2f} {mae:>8.2f} {r2:>8.4f} {ns:>12,.0f}")
print("=" * 75)

comparison_df = pd.DataFrame(comparison_data)

## 4. Analisis por Rango de RUL

El modelo debe ser preciso donde importa: cuando al motor le quedan POCOS ciclos (RUL < 30). Un error de 20 ciclos cuando RUL=100 es tolerable. Un error de 20 ciclos cuando RUL=10 puede ser fatal.

In [ ]:
# Analisis por rango para cada modelo
rul_bins = [(0, 20, 'Critico (0-20)'), (20, 50, 'Degradado (20-50)'), (50, 130, 'Sano (50-130)')]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (name, (yt, yp)) in enumerate(models.items()):
    ax = axes[idx]
    maes = []
    labels = []
    colors = [CRITICAL, DEGRADED, HEALTHY]
    
    for low, high, label in rul_bins:
        mask = (yt >= low) & (yt < high) if high < 130 else (yt >= low) & (yt <= high)
        if mask.sum() > 0:
            mae_bin = mean_absolute_error(yt[mask], yp[mask])
            maes.append(mae_bin)
            labels.append(f'{label}\n(n={mask.sum()})')
        else:
            maes.append(0)
            labels.append(f'{label}\n(n=0)')
    
    bars = ax.bar(range(len(maes)), maes, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_ylabel('MAE (ciclos)')
    
    for bar, val in zip(bars, maes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', fontweight='bold')

plt.suptitle('MAE por Rango de RUL: Donde Importa la Precision', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Distribucion Early vs Late Predictions

Cuantas predicciones son conservadoras (early) vs peligrosas (late)?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, (yt, yp)) in enumerate(models.items()):
    ax = axes[idx]
    d = yp - yt
    
    early = (d < 0).sum()
    late = (d >= 0).sum()
    
    ax.hist(d[d < 0], bins=20, alpha=0.7, color=NEUTRAL, label=f'Early: {early} ({early/len(d)*100:.0f}%)', edgecolor='white')
    ax.hist(d[d >= 0], bins=20, alpha=0.7, color=CRITICAL, label=f'Late: {late} ({late/len(d)*100:.0f}%)', edgecolor='white')
    ax.axvline(x=0, color='black', linewidth=2, linestyle='--')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('d = Predicho - Real')
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.suptitle('Distribucion de Errores: Early (seguro) vs Late (peligroso)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Sistema de Alertas

Definimos tres niveles de alerta basados en el RUL predicho:

| Nivel | RUL Predicho | Accion |
|-------|-------------|--------|
| VERDE | > 50 ciclos | Operacion normal. Monitoreo de rutina. |
| AMARILLO | 20 - 50 ciclos | Programar inspeccion. Preparar repuestos. |
| ROJO | < 20 ciclos | Retirar de servicio. Mantenimiento inmediato. |

In [ ]:
def classify_alert(rul_pred):
    """Clasificar RUL predicho en nivel de alerta."""
    if rul_pred > 50:
        return 'VERDE'
    elif rul_pred > 20:
        return 'AMARILLO'
    else:
        return 'ROJO'

# Seleccionar el mejor modelo
best_model_name = comparison_df.loc[comparison_df['RMSE'].idxmin(), 'Modelo']
best_yt, best_yp = models[best_model_name]

print(f"Modelo seleccionado: {best_model_name}")
print()

# Clasificar alertas
alerts_pred = [classify_alert(p) for p in best_yp]
alerts_real = [classify_alert(r) for r in best_yt]

# Matriz de confusion de alertas
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

alert_levels = ['ROJO', 'AMARILLO', 'VERDE']
cm = confusion_matrix(alerts_real, alerts_pred, labels=alert_levels)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=alert_levels)
disp.plot(ax=ax, cmap='YlOrRd', values_format='d')
ax.set_title(f'Matriz de Confusion de Alertas ({best_model_name})', fontsize=14, fontweight='bold')
ax.set_xlabel('Alerta Predicha')
ax.set_ylabel('Alerta Real')
plt.tight_layout()
plt.show()

print("Distribucion de alertas:")
for level in alert_levels:
    count_pred = alerts_pred.count(level)
    count_real = alerts_real.count(level)
    print(f"  {level:>10}: Predichas={count_pred}, Reales={count_real}")

## 7. Simulacion Operativa

Simulamos la operacion de un motor del test set ciclo a ciclo, mostrando como evolucionan las alertas. Esto es lo que veria un operador en un dashboard real.

In [ ]:
# Seleccionar un motor con vida suficiente para una buena visualizacion
test_unit = sorted(df_test['unit'].unique())[0]
unit_data = df_test[df_test['unit'] == test_unit].sort_values('time_cycles')

print(f"Simulacion operativa — Motor {test_unit}")
print(f"Ciclos disponibles: {len(unit_data)}")

# Predicciones ciclo a ciclo con LSTM
features = unit_data[feature_cols].values
rul_real = unit_data['RUL'].values
cycles = unit_data['time_cycles'].values

if len(features) < WINDOW_SIZE:
    pad_length = WINDOW_SIZE - len(features)
    features_padded = np.vstack([np.tile(features[0], (pad_length, 1)), features])
else:
    features_padded = features

preds = []
for i in range(WINDOW_SIZE, len(features_padded) + 1):
    seq = features_padded[i - WINDOW_SIZE:i].reshape(1, WINDOW_SIZE, -1)
    pred = lstm_model.predict(seq, verbose=0)[0, 0]
    preds.append(max(0, pred))

preds = preds[-len(cycles):]

# Visualizacion tipo dashboard
fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={'height_ratios': [3, 1]})

# Panel superior: RUL real vs predicho
ax = axes[0]
ax.plot(cycles, rul_real, color='black', linewidth=2, label='RUL Real')
ax.plot(cycles, preds, color=NEUTRAL, linewidth=2, linestyle='--', label='RUL Predicho (LSTM)')
ax.axhline(y=50, color=HEALTHY, linewidth=1.5, linestyle=':', alpha=0.8, label='Umbral VERDE/AMARILLO')
ax.axhline(y=20, color=CRITICAL, linewidth=1.5, linestyle=':', alpha=0.8, label='Umbral AMARILLO/ROJO')
ax.fill_between(cycles, 0, 20, alpha=0.1, color=CRITICAL)
ax.fill_between(cycles, 20, 50, alpha=0.1, color=DEGRADED)
ax.fill_between(cycles, 50, max(max(rul_real), max(preds)) + 10, alpha=0.1, color=HEALTHY)
ax.set_title(f'Simulacion Operativa — Motor {test_unit}', fontsize=14, fontweight='bold')
ax.set_ylabel('RUL (ciclos restantes)')
ax.legend(loc='upper right')

# Panel inferior: nivel de alerta
ax = axes[1]
alert_colors = {'VERDE': HEALTHY, 'AMARILLO': DEGRADED, 'ROJO': CRITICAL}
for i, pred in enumerate(preds):
    alert = classify_alert(pred)
    ax.bar(cycles[i], 1, color=alert_colors[alert], width=1.0, edgecolor='none')

ax.set_title('Nivel de Alerta (basado en prediccion)', fontsize=12, fontweight='bold')
ax.set_xlabel('Ciclo')
ax.set_yticks([])
ax.set_xlim(cycles[0], cycles[-1])

# Leyenda de alertas
patches = [mpatches.Patch(color=HEALTHY, label='VERDE (>50)'),
           mpatches.Patch(color=DEGRADED, label='AMARILLO (20-50)'),
           mpatches.Patch(color=CRITICAL, label='ROJO (<20)')]
ax.legend(handles=patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Tabla de eventos criticos
print(f"BITACORA DE ALERTAS — Motor {test_unit}")
print("=" * 60)

prev_alert = None
for cycle, pred, real in zip(cycles, preds, rul_real):
    alert = classify_alert(pred)
    if alert != prev_alert:
        real_alert = classify_alert(real)
        status = "OK" if alert == real_alert else f"(Real: {real_alert})"
        print(f"  Ciclo {cycle:>3}: [{alert:>8}] RUL_pred={pred:.0f}, RUL_real={real:.0f} {status}")
        prev_alert = alert

print("=" * 60)

## 8. Resumen Ejecutivo

In [ ]:
best_row = comparison_df.loc[comparison_df['RMSE'].idxmin()]

print("=" * 70)
print("  RESUMEN EJECUTIVO — Mantenimiento Predictivo C-MAPSS FD001")
print("=" * 70)
print()
print(f"  Mejor modelo:     {best_row['Modelo']}")
print(f"  RMSE:             {best_row['RMSE']:.2f} ciclos")
print(f"  MAE:              {best_row['MAE']:.2f} ciclos")
print(f"  R2:               {best_row['R2']:.4f}")
print(f"  NASA Score:       {best_row['NASA_Score']:,.0f}")
print()
print("  MODELOS EVALUADOS:")
for _, row in comparison_df.iterrows():
    print(f"    {row['Modelo']:<20} RMSE={row['RMSE']:.2f}  NASA={row['NASA_Score']:,.0f}")
print()
print("  SISTEMA DE ALERTAS:")
print("    VERDE:    RUL > 50 ciclos  -> Operacion normal")
print("    AMARILLO: 20 < RUL < 50   -> Programar inspeccion")
print("    ROJO:     RUL < 20 ciclos  -> Mantenimiento inmediato")
print()
print("  LIMITACIONES:")
print("    - Entrenado con 100 motores (FD001) — una condicion operativa")
print("    - No probado con datos reales de sensores (solo simulacion)")
print("    - La LSTM requiere ventana de 30 ciclos para predecir")
print()
print("=" * 70)

## 9. Arquitectura de Produccion (Conceptual)

```
Sensores del Motor
       |
       v
[Ingesta en Tiempo Real]  -->  Buffer de Ultimos 30 Ciclos
       |
       v
[Pipeline de Features]    -->  Normalizacion + Rolling Stats + Tendencias
       |
       v
[Modelo LSTM]             -->  RUL Predicho
       |
       v
[Motor de Alertas]        -->  VERDE / AMARILLO / ROJO
       |
       v
[Dashboard Operativo]     -->  Notificacion a equipo de mantenimiento
```

### Componentes necesarios
1. **Ingesta:** Streaming de datos de sensores (Kafka, MQTT)
2. **Pipeline:** Preprocesamiento en tiempo real (Spark Streaming, Python)
3. **Modelo:** Servido via API REST (TensorFlow Serving, FastAPI)
4. **Alertas:** Reglas de negocio + notificaciones (PagerDuty, email)
5. **Dashboard:** Visualizacion en tiempo real (Grafana, Streamlit)
6. **Retraining:** Pipeline de reentrenamiento periodico con nuevos datos de fallo

## Conclusion Final

Este proyecto demuestra un pipeline completo de **Mantenimiento Predictivo**:

1. **Ingesta:** Datos crudos de la NASA -> RUL calculado con piecewise linear
2. **EDA:** 21 sensores analizados, 7 constantes eliminados
3. **Feature Engineering:** Rolling stats, tendencias, normalizacion por motor
4. **Modelos Clasicos:** Random Forest, XGBoost, SVR comparados
5. **Deep Learning:** LSTM para capturar patrones secuenciales
6. **Produccion:** Sistema de alertas verde/amarillo/rojo demostrado

**La diferencia entre mantenimiento reactivo y predictivo es la diferencia entre apagar incendios y prevenirlos.**